# 05 · Feature engineering for `(T,225)` arrays

**Adapted:** Reads the delivered `.npy` arrays, from the current `(T,225)` sequence format. Builds fixed-length temporal pooled coordinate and velocity summaries. Per-feature imputation/scaling is intentionally deferred to model pipelines so it is fit on training folds only. Shoulder normalization is applied per sequence using pose landmarks 11/12.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().resolve()
while not (ROOT/'data/processed/WLASL100/all_metadata.csv').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
DATA=ROOT/'data/processed/WLASL100'
if not (DATA/'all_metadata.csv').exists(): raise FileNotFoundError('Open notebook from the sign-language-recognition repository')
import numpy as np
split=pd.read_csv(DATA/'split_v2.csv',dtype={'video_id':str})
meta=pd.read_csv(DATA/'all_metadata.csv',dtype={'video_id':str}); meta['video_id']=meta.video_id.str.zfill(5)
source={}
for s in ('landmarks_repo','landmarks_kaggle'):
 for p in (DATA/s).glob('*.npy'): source[p.stem]=p

def make_features(a,steps=16):
 a=a.astype(np.float32,copy=True)
 # per-frame shoulder-centred, shoulder-width scaled coordinates
 pose=a[:,:99].reshape(-1,33,3); shoulders=pose[:,[11,12],:]
 center=np.nanmean(shoulders,axis=1); width=np.linalg.norm(shoulders[:,0]-shoulders[:,1],axis=1)
 scale=np.where(np.isfinite(width)&(width>1e-5),width,1).astype(np.float32)
 xyz=a.reshape(-1,75,3); xyz=(xyz-center[:,None,:])/scale[:,None,None]
 # Resample sequence and summarize 16 temporal bins; include first differences.
 idx=np.linspace(0,len(xyz)-1,steps).round().astype(int); seq=xyz[idx]
 vel=np.diff(seq,axis=0,prepend=seq[:1])
 return np.concatenate([np.nanmean(seq,axis=0).ravel(),np.nanstd(seq,axis=0).ravel(),np.nanmean(vel,axis=0).ravel(),np.nanstd(vel,axis=0).ravel()]).astype(np.float32)
rows=[]; X=[]
for r in split.itertuples():
 vid=str(r.video_id).zfill(5)
 if vid not in source: continue
 X.append(make_features(np.load(source[vid])))
 rows.append({'video_id':vid,'gloss':r.gloss,'signer_id':str(r.signer_id),'split':r.split})
features=pd.DataFrame(rows); matrix=np.stack(X)
np.savez_compressed(DATA/'features_v1.npz',X=matrix,video_id=features.video_id.to_numpy(dtype='U5'))
features.to_csv(DATA/'feature_manifest_v1.csv',index=False)
print('feature matrix:',matrix.shape,'; saved features_v1.npz and feature_manifest_v1.csv')
